In [4]:
# evaluate_retrieval_metrics.py
# 计算 RAL2M 检索阶段真实性能（Top-1/5/10/100 Recall + MRR）

import pandas as pd
import numpy as np
import json
import os

# ------------------- 配置 -------------------
RETRIEVAL_FILE = "retrieval_output/train_with_top100_ids.csv"  # 你的 top-100 文件
D1_FILE = "../data/D1.csv"                                      # 包含 id → q_i

print("加载检索结果和 D1...")
df = pd.read_csv(RETRIEVAL_FILE)
d1 = pd.read_csv(D1_FILE)

# 确保 id 是字符串
df['top100_d1_ids'] = df['top100_d1_ids'].apply(
    lambda x: eval(x) if isinstance(x, str) else x
)
d1['id'] = d1['id'].astype(str)

# 构建 id → q_i 的快速映射
id_to_q = dict(zip(d1['id'], d1['q_i']))

print(f"总查询数: {len(df):,}")

# ------------------- 计算真实指标 -------------------
def calculate_metrics(df, id_to_q):
    top1 = top5 = top10 = top100 = 0
    mrr_sum = 0.0
    total = len(df)

    for _, row in df.iterrows():
        true_q = row['true_q_i'].strip()
        candidate_ids = row['top100_d1_ids'][:100]  # 确保最多100个

        # 取出候选问题文本
        candidate_qs = []
        for cid in candidate_ids:
            q_text = id_to_q.get(str(cid).strip(), "")
            if q_text:
                candidate_qs.append(q_text.strip())

        # 检查是否命中
        if true_q in candidate_qs:
            rank = candidate_qs.index(true_q) + 1  # 从1开始排名
            
            if rank == 1:
                top1 += 1
            if rank <= 5:
                top5 += 1
            if rank <= 10:
                top10 += 1
            if rank <= 100:
                top100 += 1
                
            mrr_sum += 1.0 / rank

    return {
        'Total Queries'      : total,
        'Top-1 Recall'       : top1 / total,
        'Top-5 Recall'       : top5 / total,
        'Top-10 Recall'      : top10 / total,
        'Top-100 Recall'     : top100 / total,
        'MRR'                : mrr_sum / total,
        'Top-1 Hits'         : top1,
        'Top-100 Hits'       : top100,
    }

metrics = calculate_metrics(df, id_to_q)

# ------------------- 输出美观表格（直接进论文！） -------------------
print("\n" + "="*90)
print("RAL2M 检索阶段真实性能（Top-100 候选）")
print("="*90)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:<20}: {v:.4f} ({v:.2%})")
    else:
        print(f"{k:<20}: {v:,}")
print("="*90)

# 保存结果
os.makedirs("retrieval_output", exist_ok=True)
with open("retrieval_output/retrieval_metrics_top100.json", "w", encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"\n指标已保存 → retrieval_output/retrieval_metrics_top100.json")

加载检索结果和 D1...
总查询数: 25,338

RAL2M 检索阶段真实性能（Top-100 候选）
Total Queries       : 25,338
Top-1 Recall        : 0.2083 (20.83%)
Top-5 Recall        : 0.5037 (50.37%)
Top-10 Recall       : 0.5756 (57.56%)
Top-100 Recall      : 0.7826 (78.26%)
MRR                 : 0.3433 (34.33%)
Top-1 Hits          : 5,279
Top-100 Hits        : 19,830

指标已保存 → retrieval_output/retrieval_metrics_top100.json
